In [ ]:
import os
import cv2
import numpy as np
from PIL import Image, ImageTk
from pathlib import Path

import pandas as pd
import mediapipe as mp
import matplotlib.pyplot as plt
import seaborn as sns

import tkinter as tk
from tkinter import filedialog

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import  confusion_matrix,roc_curve, auc,classification_report
from sklearn.preprocessing import LabelEncoder, LabelBinarizer

import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import load_model,Model
from tensorflow.keras.layers import Flatten, Dense 
from tensorflow.keras.applications import VGG16

In [ ]:
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils 
mp_drawing_styles = mp.solutions.drawing_styles
hands = mp_hands.Hands(static_image_mode=True, min_detection_confidence=0.9)

mpPose = mp.solutions.pose
pose = mpPose.Pose(
    static_image_mode=True,
    min_detection_confidence=0.3,
    model_complexity=2
)

GET DATASET

In [ ]:

SIZE = 128

def get_dataset(dataset):
    X = []
    y = []

    image_dir = Path(dataset)
    # Get filepaths and labels
    filepaths = list(image_dir.glob(r'**/*.JPG')) + list(image_dir.glob(r'**/*.jpeg')) + list(image_dir.glob(r'**/*.png'))

    labels = list(map(lambda x: os.path.split(os.path.split(x)[0])[1], filepaths))
    for file in filepaths[:]:
        img = Image.open(file).resize((SIZE,SIZE))
        img_rgb = img.convert('RGB')
        img = np.asarray(img_rgb) / 255.0

        X.append(img)

    for label in labels[:]:
        y.append(label)

    return X,y
X, y = get_dataset(image_dir)


In [ ]:
def show_images(image_dir):
    for i in sorted(os.listdir(image_dir)):
        if i == '.DS_Store':
            pass
        else:
            for j in os.listdir(os.path.join(image_dir,i))[0:1]:
                img = cv2.imread(os.path.join(image_dir,i,j))
                img_rgb = cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

                results = hands.process(img_rgb)
                if results.multi_hand_landmarks:
                    for hand_landmarks in results.multi_hand_landmarks:
                        mp_drawing.draw_landmarks(
                            img_rgb, # img to draw
                            hand_landmarks,
                            mp_hands.HAND_CONNECTIONS,
                            mp_drawing_styles.get_default_hand_landmarks_style(),
                            mp_drawing_styles.get_default_hand_connections_style()
                        )

                plt.figure()
                plt.title(i)
                plt.imshow(img_rgb)
    plt.show()
show_images(image_dir)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(np.array(X), np.array(y), test_size=0.4, random_state=22, shuffle=True)


In [ ]:
print(f'X Train shape: {X_train.shape}')
print(f'X Test shape: {X_test.shape}')
print(f'Y Train shape: {y_train.shape}')
print(f'Y Test shape: {y_test.shape}')


In [ ]:
# Encode labels
encoder = LabelEncoder()
encoder.fit(y_train)

y_train = encoder.transform(y_train)
y_test = encoder.transform(y_test)

y_train = to_categorical(y_train, len(encoder.classes_))
y_test = to_categorical(y_test, len(encoder.classes_))

In [ ]:
labels = encoder.classes_
num_classes = len(labels)

In [ ]:
print("Shape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)


VGG16

In [ ]:
def vgg16():
        
    weight_path1 = 'D:/Private/Project/kaggle/sign_detection/vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5'

    pretrained_model_1 = VGG16(weights = weight_path1, include_top=False, input_shape=(SIZE, SIZE, 3))
  
    base_model = pretrained_model_1 # Topless
    # Add top layer
    x = base_model.output
    x = Flatten()(x)
    predictions = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=predictions)
    # Train top layer
    for layer in base_model.layers:
        layer.trainable = False
        
    model.compile(loss='categorical_crossentropy', 
                    optimizer='adam', 
                    metrics=['accuracy'])
    return model


model_vgg16 = vgg16()
model_vgg16.summary()

history_vgg16  = model_vgg16.fit(X_train,y_train, epochs=5, validation_data=(X_test, y_test))
model_vgg16.save('model_vgg16.keras')

Prediction

In [ ]:
def predict_sign(model, path):
    img = Image.open(path).resize((SIZE,SIZE))
    img_rgb = img.convert('RGB')
    img = np.asarray(img_rgb) / 255.0
    img = np.expand_dims(img, axis=0)
    prediction = model.predict(img)
    predicted = labels[np.argmax(prediction)]
    return predicted

path = 'D:/Private/Project/kaggle/sign_detection/sign_image/Y/hand_46.jpg'
predict_sign(model_vgg16, path)

GET Labels

In [ ]:
labels = []
for f in os.listdir(image_dir):
    labels.append(f)
print(labels)

Evaluation

In [ ]:

model = load_model('model_vgg16.keras')
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {test_acc}")
print(f"Test loss: {test_loss}")


In [ ]:
labels = ['A', 'B', 'C', 'D', 'Delete', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 
          'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'Space', 'T', 'U', 'V', 
          'W', 'X', 'Y', 'Z']

y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

report = classification_report(y_test_classes, y_pred_classes, target_names=labels, output_dict=True)

df_report = pd.DataFrame(report).transpose()

df_report_filtered = df_report[[ "precision", "recall", "f1-score"]]
df_report_filtered.to_xlsx("classification_report.csv", index=True)

print(df_report_filtered)

In [ ]:
#Confusion matrix
conf_matrix = confusion_matrix(y_test_classes, y_pred_classes)
labels = ['A', 'B', 'C', 'D', 'Delete', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'Space', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']

plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, cbar=False)

plt.xticks(rotation=45, ha='right', fontsize=10)  
plt.yticks(rotation=0, fontsize=10)  
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title('Confusion Matrix', fontsize=14)

plt.tight_layout() 
plt.savefig('confusion_matrix.png', bbox_inches='tight')
plt.show()

In [ ]:
#ROC 
lb = LabelBinarizer()
y_test_bin = lb.fit_transform(y_test)

plt.figure(figsize=(10, 8))
for i in range(y_test_bin.shape[1]):
    fpr, tpr, thresholds = roc_curve(y_test_bin[:, i], y_pred[:, i])  
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'ROC curve for label {labels[i]} (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')  # Đường chéo

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves for Each Label')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10)

plt.tight_layout()  
plt.show()


Test Cam

In [ ]:

# Define labels for gestures
labels = ['A', 'B', 'C', 'D', 'Delete', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 
          'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'Space', 'T', 'U', 'V', 
          'W', 'X', 'Y', 'Z']

class SignLanguageApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Sign Language to Text")
        self.root.geometry("600x600")
        self.root.resizable(False, False)
        self.root.config(bg="#f0f0f0")

        # Create GUI components
        self.create_gui()

        # Video capture setup
        self.cap = cv2.VideoCapture(0)
        self.previous_label = ""
        self.frame_count = 0
        self.required_frames = 8

        self.run_camera()

    def create_gui(self):
     
        self.video_label = tk.Label(self.root, bg="black", width=500, height=350)
        self.video_label.pack(pady=10)

    
        self.text_box = tk.Text(self.root, height=3, width=60, font=("Arial", 14), bd=2, relief="solid")
        self.text_box.pack(pady=10)

  
        button_frame = tk.Frame(self.root, bg="#f0f0f0")
        button_frame.pack(pady=10)


        tk.Button(button_frame, text="Clear All", bg="#FFCC00", font=("Arial", 12), width=20,
                  command=self.clear_text).grid(row=0, column=0, padx=10, pady=5)


        tk.Button(button_frame, text="Save to a Text File", bg="#4CAF50", font=("Arial", 12), width=20,
                  command=self.save_to_file).grid(row=0, column=1, padx=10, pady=5)

        # Quit button
        tk.Button(button_frame, text="Quit", bg="#F44336", font=("Arial", 12), width=15,
                  command=self.quit_app).grid(row=0, column=2, padx=10, pady=5)

    def run_camera(self):
        ret, frame = self.cap.read()
        if ret:
            frame = cv2.flip(frame, 1)
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


            with mp_hands.Hands(static_image_mode=False, max_num_hands=1, 
                                min_detection_confidence=0.8, 
                                min_tracking_confidence=0.8) as hands:
                result = hands.process(rgb_frame)
                if result.multi_hand_landmarks:
                    for hand_landmarks in result.multi_hand_landmarks:
                        mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS,
                                                  mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=2),
                                                  mp_drawing.DrawingSpec(color=(255, 165, 0), thickness=2))

                        x_min, y_min, x_max, y_max = self.get_hand_bbox(hand_landmarks, frame)
                        hand_roi = frame[y_min:y_max, x_min:x_max]
                        if hand_roi.size > 0:
                            hand_resized = cv2.resize(hand_roi, (128, 128))
                            hand_resized = hand_resized / 255.0
                            hand_resized = np.expand_dims(hand_resized, axis=0)

                            prediction = model.predict(hand_resized)[0]
                            predicted_label = labels[np.argmax(prediction)]

                    
                            self.handle_prediction(predicted_label, x_min, y_min, frame)

            img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            img_tk = ImageTk.PhotoImage(image=img)
            self.video_label.imgtk = img_tk
            self.video_label.configure(image=img_tk)

        self.root.after(10, self.run_camera)

    def get_hand_bbox(self, hand_landmarks, frame):
        h, w, _ = frame.shape
        x_min, y_min, x_max, y_max = w, h, 0, 0
        for landmark in hand_landmarks.landmark:
            x, y = int(landmark.x * w), int(landmark.y * h)
            x_min, y_min = min(x, x_min), min(y, y_min)
            x_max, y_max = max(x, x_max), max(y, y_max)

        padding = 20
        x_min, y_min = max(0, x_min - padding), max(0, y_min - padding)
        x_max, y_max = min(w, x_max + padding), min(h, y_max + padding)
        return x_min, y_min, x_max, y_max

    def handle_prediction(self, predicted_label, x_min, y_min, frame):
        if predicted_label == self.previous_label:
            self.frame_count += 1
        else:
            self.frame_count = 1
            self.previous_label = predicted_label

        if self.frame_count >= self.required_frames:
            self.append_to_text(predicted_label)
            self.frame_count = 0

        cv2.putText(frame, predicted_label, (x_min, y_min - 10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)

    def append_to_text(self, predicted_label):
        current_text = self.text_box.get(1.0, tk.END).strip()
        self.text_box.delete(1.0, tk.END)

        if predicted_label == "Space":
            self.text_box.insert(tk.END, current_text + " ")
        elif predicted_label == "Delete":
            self.text_box.insert(tk.END, current_text[:-1])
        else:
            self.text_box.insert(tk.END, current_text + predicted_label)

    def clear_text(self):
        self.text_box.delete(1.0, tk.END)

    def save_to_file(self):
        file_path = filedialog.asksaveasfilename(defaultextension=".txt", 
                                                 filetypes=[("Text files", "*.txt")])
        if file_path:
            with open(file_path, 'w') as file:
                file.write(self.text_box.get(1.0, tk.END).strip())
            print(f"Text saved to {file_path}")

    def quit_app(self):
        self.cap.release()
        cv2.destroyAllWindows()
        self.root.destroy()

# Run the application
if __name__ == "__main__":
    root = tk.Tk()
    app = SignLanguageApp(root)
    root.mainloop()
